In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 1  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start, utc=True)]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,confidence,threatAssessRating,threatAssessConfidence,...,tags.data,rating,description,associatedGroups.data,hostName,dnsActive,whoisActive,address,indicator,sources
2,6755399487000198,2026-01-09T15:38:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-18T11:31:01Z,73.0,2.0,41.33,...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",3.0,RITM0637479,NaN,NaN,NaN,NaN,NaN,199.45.154.137,HTOC Org
3,6755399465229342,2025-07-28T17:16:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-18T11:22:50Z,48.0,1.5,24.50,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,198.235.24.111,"DHS CISCP, HTOC Org"
4,6755399465229552,2025-07-28T17:34:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-18T11:22:49Z,48.0,3.0,48.00,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,64.62.156.112,HTOC Org
5,6755399465229530,2025-07-28T17:34:14Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-18T11:22:49Z,49.0,3.0,49.00,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,64.62.156.80,HTOC Org
6,6755399465229511,2025-07-28T17:34:02Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-18T11:22:49Z,48.0,1.5,24.50,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,205.210.31.212,"DHS CISCP, HTOC Org"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1252,5629499583003490,2026-01-12T13:46:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-11T00:23:37Z,65.0,5.0,65.00,...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...",5.0,Summary\nRussian actors use obfuscated infrast...,"[{'id': 5629499569000349, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,159.203.24.4,HTOC Org
1253,5629499561376181,2025-07-28T17:16:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-11T00:23:37Z,49.0,1.5,25.00,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,205.210.31.224,"DHS CISCP, HTOC Org"
1254,5629499561376177,2025-07-28T17:15:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-11T00:23:35Z,49.0,1.5,29.00,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,198.235.24.96,"DHS CISCP, HTOC Org"
1255,5629499561376912,2025-07-28T17:34:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-11T00:23:34Z,49.0,1.5,24.50,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",3.0,TASK0902923,NaN,NaN,NaN,NaN,NaN,65.49.1.222,HTOC Org


In [3]:
import pandas as pd

# Load the Excel file
file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(file_path)

# Keep only indicators that are also in observed_src
_indicator_col = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
if _indicator_col is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")

_observed_indicators = set(observed_src["indicator"].dropna().astype(str))
df = df[df[_indicator_col].astype(str).isin(_observed_indicators)].copy()

# Update df's Last Observed from observed_src.lastObserved
_last_observed_col = next(
    (
        c
        for c in [
            "Last Observed",
            "lastObserved",
            "LastObserved",
            "last_observed",
            "LAST OBSERVED",
        ]
        if c in df.columns
    ),
    None,
)
if _last_observed_col is None:
    raise KeyError(f"Could not find 'Last Observed' column in df. Columns: {list(df.columns)}")

_last_obs_by_indicator = (
    observed_src.dropna(subset=["indicator"])
    .assign(
        indicator=lambda d: d["indicator"].astype(str),
        lastObserved=lambda d: pd.to_datetime(d["lastObserved"], utc=True, errors="coerce"),
    )
    .groupby("indicator", dropna=False)["lastObserved"]
    .max()
)

# Ensure df's last observed column is datetime-like, then overwrite for matches
_df_ind = df[_indicator_col].astype(str)
df[_last_observed_col] = pd.to_datetime(df[_last_observed_col], utc=True, errors="coerce")
df[_last_observed_col] = _df_ind.map(_last_obs_by_indicator).combine_first(df[_last_observed_col])

df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
4,103.120.116.162,2026-03-18 00:00:00+00:00,Address,48,3,0.997370,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9385644,NaN,740,870,364,medium,[2026-03-16] Severity: medium. VT score: 6. To...
7,103.143.10.79,2026-03-18 00:00:00+00:00,Address,17,3,0.999068,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,770,764,673,high,[2026-03-16] Severity: high. VT score: 17. Top...
9,103.203.59.0,2026-03-18 00:00:00+00:00,Address,282,3,0.984548,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",NaN,NaN,180,368,90,low,[2026-03-16] Severity: low. VT score: 1. Top d...
13,103.243.26.49,2026-03-18 00:00:00+00:00,Address,55,3,0.996986,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9375302,NaN,180,469,351,medium,[2026-03-16] Severity: medium. VT score: 6. To...
16,104.128.161.233,2026-03-18 00:00:00+00:00,Address,299,5,0.983616,0,0,"DHA, NIH, VA",NaN,"Cozy Bear, Fancy Bear",180,330,319,medium,[2026-03-16] Severity: medium. VT score: 5. To...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1368,link.storjshare.io,2026-03-18 00:00:00+00:00,Host,4,4,0.999781,0,0,"IHS, NIH",NaN,Banished Kitten,180,469,161,low,[2026-03-16] Severity: low. VT score not avail...
1391,194.33.45.162,2026-03-17 00:00:00+00:00,Address,35,5,0.998082,0,0,"NIH, OS",NaN,"Diamond Sleet, Emerald Sleet, Famous Chollima,...",180,433,439,medium,[2026-03-11] Severity: medium. VT score: 10. T...
1723,185.248.85.49,2026-03-18 00:00:00+00:00,Address,23,5,0.998740,0,0,DHA,NaN,UNC5537,170,585,233,medium,[2026-03-06] Severity: medium. VT score: 4. To...
1849,185.225.28.5,2026-03-17 00:00:00+00:00,Address,9,5,0.999507,0,0,"FDA, HRSA, OS, VA",NaN,NaN,180,590,199,low,Severity: low. VT score: 1. Top drivers: TC ra...


In [4]:
# Filter last 24h results to indicators seen at more than one partner
last_24h_multiple_partners = df[df['Partners'].str.contains(',', na=False)]

last_24h_multiple_partners

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
4,103.120.116.162,2026-03-18 00:00:00+00:00,Address,48,3,0.997370,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9385644,NaN,740,870,364,medium,[2026-03-16] Severity: medium. VT score: 6. To...
7,103.143.10.79,2026-03-18 00:00:00+00:00,Address,17,3,0.999068,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,770,764,673,high,[2026-03-16] Severity: high. VT score: 17. Top...
9,103.203.59.0,2026-03-18 00:00:00+00:00,Address,282,3,0.984548,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",NaN,NaN,180,368,90,low,[2026-03-16] Severity: low. VT score: 1. Top d...
13,103.243.26.49,2026-03-18 00:00:00+00:00,Address,55,3,0.996986,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9375302,NaN,180,469,351,medium,[2026-03-16] Severity: medium. VT score: 6. To...
16,104.128.161.233,2026-03-18 00:00:00+00:00,Address,299,5,0.983616,0,0,"DHA, NIH, VA",NaN,"Cozy Bear, Fancy Bear",180,330,319,medium,[2026-03-16] Severity: medium. VT score: 5. To...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,180.167.128.202,2026-03-18 00:00:00+00:00,Address,5,3,0.999726,0,0,"CDC, CMS, DHA, FDA, HRSA, IHS, NIH, OS",Incident:INC9444954,NaN,180,590,337,medium,[2026-03-16] Severity: medium. VT score: 5. To...
1368,link.storjshare.io,2026-03-18 00:00:00+00:00,Host,4,4,0.999781,0,0,"IHS, NIH",NaN,Banished Kitten,180,469,161,low,[2026-03-16] Severity: low. VT score not avail...
1391,194.33.45.162,2026-03-17 00:00:00+00:00,Address,35,5,0.998082,0,0,"NIH, OS",NaN,"Diamond Sleet, Emerald Sleet, Famous Chollima,...",180,433,439,medium,[2026-03-11] Severity: medium. VT score: 10. T...
1849,185.225.28.5,2026-03-17 00:00:00+00:00,Address,9,5,0.999507,0,0,"FDA, HRSA, OS, VA",NaN,NaN,180,590,199,low,Severity: low. VT score: 1. Top drivers: TC ra...


In [5]:
# Filter multi-partner, last-24h indicators to VT score >= 10 based on Explanation text
vt_scores = last_24h_multiple_partners['Explanation'].str.extract(r'VT score:\s*(\d+)', expand=False)
vt_scores = pd.to_numeric(vt_scores, errors='coerce')

last_24h_multi_partners_vt15 = last_24h_multiple_partners[vt_scores >= 15]

last_24h_multi_partners_vt15

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
7,103.143.10.79,2026-03-18 00:00:00+00:00,Address,17,3,0.999068,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,770,764,673,high,[2026-03-16] Severity: high. VT score: 17. Top...
65,123.58.207.81,2026-03-18 00:00:00+00:00,Address,215,3,0.988219,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9185304,NaN,380,439,619,high,[2026-03-16] Severity: high. VT score: 15. Top...
93,138.59.233.5,2026-03-18 00:00:00+00:00,Address,71,3,0.996110,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,540,600,626,high,[2026-03-16] Severity: high. VT score: 15. Top...
95,139.19.117.131,2026-03-18 00:00:00+00:00,Address,159,3,0.991288,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,790,721,650,high,[2026-03-16] Severity: high. VT score: 16. Top...
216,167.94.138.124,2026-03-18 00:00:00+00:00,Address,120,3,0.993425,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,700,693,536,high,[2026-03-16] Severity: high. VT score: 16. Top...
318,184.105.247.252,2026-03-18 00:00:00+00:00,Address,91,3,0.995014,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,290,471,551,high,[2026-03-16] Severity: high. VT score: 15. Top...
328,185.156.73.233,2026-03-18 00:00:00+00:00,Address,13,3,0.999288,0,0,"NIH, OS",Incident:INC9373125;Incident:INC9373119,NaN,570,664,539,high,[2026-03-16] Severity: high. VT score: 15. Top...
347,185.220.101.37,2026-03-17 00:00:00+00:00,Address,139,2,0.992384,0,0,"CDC, CMS, FDA, HHS, HRSA, NIH, OS, VA",NaN,NaN,720,582,499,medium,[2026-03-16] Severity: medium. VT score: 17. T...
483,193.32.162.145,2026-03-18 00:00:00+00:00,Address,153,3,0.991616,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,810,784,583,high,[2026-03-16] Severity: high. VT score: 16. Top...
534,198.235.24.194,2026-03-18 00:00:00+00:00,Address,232,3,0.987288,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-16] Severity: high. VT score: 15. Top...


In [6]:
# Keep only high or critical indicators from the VT>=10, multi-partner, last-24h set
final_indicators = last_24h_multi_partners_vt15[last_24h_multi_partners_vt15['Severity'].isin(['high', 'critical'])]

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
7,103.143.10.79,2026-03-18 00:00:00+00:00,Address,17,3,0.999068,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,770,764,673,high,[2026-03-16] Severity: high. VT score: 17. Top...
65,123.58.207.81,2026-03-18 00:00:00+00:00,Address,215,3,0.988219,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9185304,NaN,380,439,619,high,[2026-03-16] Severity: high. VT score: 15. Top...
93,138.59.233.5,2026-03-18 00:00:00+00:00,Address,71,3,0.996110,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,540,600,626,high,[2026-03-16] Severity: high. VT score: 15. Top...
95,139.19.117.131,2026-03-18 00:00:00+00:00,Address,159,3,0.991288,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,790,721,650,high,[2026-03-16] Severity: high. VT score: 16. Top...
216,167.94.138.124,2026-03-18 00:00:00+00:00,Address,120,3,0.993425,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,700,693,536,high,[2026-03-16] Severity: high. VT score: 16. Top...
318,184.105.247.252,2026-03-18 00:00:00+00:00,Address,91,3,0.995014,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,290,471,551,high,[2026-03-16] Severity: high. VT score: 15. Top...
328,185.156.73.233,2026-03-18 00:00:00+00:00,Address,13,3,0.999288,0,0,"NIH, OS",Incident:INC9373125;Incident:INC9373119,NaN,570,664,539,high,[2026-03-16] Severity: high. VT score: 15. Top...
483,193.32.162.145,2026-03-18 00:00:00+00:00,Address,153,3,0.991616,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,810,784,583,high,[2026-03-16] Severity: high. VT score: 16. Top...
534,198.235.24.194,2026-03-18 00:00:00+00:00,Address,232,3,0.987288,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-16] Severity: high. VT score: 15. Top...
574,198.235.24.68,2026-03-18 00:00:00+00:00,Address,232,3,0.987288,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,510,453,511,high,[2026-03-16] Severity: high. VT score: 15. Top...


In [7]:
import pandas as pd

# Helper to see if an indicator has an I&W tag
def has_iw(tags_value):
    """
    tags_value is typically a list of dicts from ThreatConnect, e.g.:
    [{'name': 'I&W'}, {'name': 'something else'}, ...]
    """
    if tags_value is None or (isinstance(tags_value, float) and pd.isna(tags_value)):
        return False

    if not isinstance(tags_value, (list, tuple)):
        return False

    for t in tags_value:
        try:
            if isinstance(t, dict):
                name = str(t.get('name', '')).strip()
            else:
                name = str(t).strip()

            if name.lower() in {"i&w", "i & w", "iw"}:
                return True
        except Exception:
            continue
    return False

# 1) Add has_iw flag to observed_src if tags.data exists
if 'tags.data' in observed_src.columns:
    observed_src['has_iw'] = observed_src['tags.data'].apply(has_iw)
else:
    observed_src['has_iw'] = False

# 2) Collapse to one flag per indicator
iw_per_indicator = (
    observed_src.groupby('indicator', dropna=False)['has_iw']
    .max()  # any True -> True
    .reset_index()
    .rename(columns={'indicator': 'Indicator', 'has_iw': 'Reported I&W?_raw'})
)

# 3) Drop ANY existing Reported I&W? variants (_x, _y, etc.)
cols_to_drop = [c for c in final_indicators.columns if c.startswith('Reported I&W?')]
final_indicators = final_indicators.drop(columns=cols_to_drop, errors='ignore')

# 4) Merge once, with a temporary raw boolean column
final_indicators = final_indicators.merge(
    iw_per_indicator,
    on='Indicator',
    how='left'
)

# 5) Convert to Yes/No, defaulting missing to 'No'
final_indicators['Reported I&W?'] = (
    final_indicators['Reported I&W?_raw']
    .fillna(False)
    .map({True: 'Yes', False: 'No'})
)

# 6) Drop the temporary raw column
final_indicators = final_indicators.drop(columns=['Reported I&W?_raw'])

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation,Reported I&W?
0,103.143.10.79,2026-03-18 00:00:00+00:00,Address,17,3,0.999068,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9426960,NaN,770,764,673,high,[2026-03-16] Severity: high. VT score: 17. Top...,No
1,123.58.207.81,2026-03-18 00:00:00+00:00,Address,215,3,0.988219,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9185304,NaN,380,439,619,high,[2026-03-16] Severity: high. VT score: 15. Top...,No
2,138.59.233.5,2026-03-18 00:00:00+00:00,Address,71,3,0.996110,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9346798,NaN,540,600,626,high,[2026-03-16] Severity: high. VT score: 15. Top...,No
3,139.19.117.131,2026-03-18 00:00:00+00:00,Address,159,3,0.991288,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9267413,NaN,790,721,650,high,[2026-03-16] Severity: high. VT score: 16. Top...,No
4,167.94.138.124,2026-03-18 00:00:00+00:00,Address,120,3,0.993425,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,700,693,536,high,[2026-03-16] Severity: high. VT score: 16. Top...,No
5,184.105.247.252,2026-03-18 00:00:00+00:00,Address,91,3,0.995014,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9332624,NaN,290,471,551,high,[2026-03-16] Severity: high. VT score: 15. Top...,No
6,185.156.73.233,2026-03-18 00:00:00+00:00,Address,13,3,0.999288,0,0,"NIH, OS",Incident:INC9373125;Incident:INC9373119,NaN,570,664,539,high,[2026-03-16] Severity: high. VT score: 15. Top...,No
7,193.32.162.145,2026-03-18 00:00:00+00:00,Address,153,3,0.991616,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",Incident:INC9263763,NaN,810,784,583,high,[2026-03-16] Severity: high. VT score: 16. Top...,No
8,198.235.24.194,2026-03-18 00:00:00+00:00,Address,232,3,0.987288,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,288,504,high,[2026-03-16] Severity: high. VT score: 15. Top...,No
9,198.235.24.68,2026-03-18 00:00:00+00:00,Address,232,3,0.987288,0,0,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",NaN,NaN,510,453,511,high,[2026-03-16] Severity: high. VT score: 15. Top...,No


In [31]:
from datetime import datetime

# Build dated output path
today_str = datetime.today().strftime('%Y%m%d')  # e.g. 20260316
output_path = rf"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\ThreatAssessI_W\ThreatAssessI_W_{today_str}.xlsx"

# Excel can't write timezone-aware datetimes; strip tz info before export
_dt_tz_cols = final_indicators.select_dtypes(include=["datetimetz"]).columns
for _c in _dt_tz_cols:
    final_indicators[_c] = final_indicators[_c].dt.tz_convert(None)

# Write to Excel
final_indicators.to_excel(output_path, index=False)

output_path

'Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\ThreatAssessI_W\\ThreatAssessI_W_20260318.xlsx'